# Financial MCQ Prompt Engineering — Groq API (qwen/qwen3-32b)

Same 6 prompt strategies as the local-GPU study, but inference runs through the **Groq API**
using 9 pooled keys with per-key RPM / RPD / TPM tracking.

| Strategy | Inference method | Key idea |
|---|---|---|
| `baseline` | API generation | Letter-only zero-shot |
| `few_shot` | API generation | 3 in-context examples |
| `cot` | API generation | "Think step by step" |
| `role` | API generation | Expert CFA persona |
| `self_consistency` | API generation × SC_SAMPLES + vote | Majority vote |
| `meta` | API generation | Explicit elimination strategy |

**Total rows**: exactly **2500** (hindi_finance capped so the sum equals 2500).

**Model**: `qwen/qwen3-32b` via Groq  
**Keys**: 9 keys, each limited to RPM=60, RPD=1000, TPM=6000

## Cell 0 — Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

import random

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Target total rows across all datasets
TOTAL_ROWS_TARGET = 2500
# hindi_finance will be capped so sum(all datasets) == TOTAL_ROWS_TARGET
# All other datasets are loaded in full.

STRATEGIES = [
    "baseline",          # zero-shot, letter-only
    "few_shot",          # 3 in-context examples
    "cot",               # chain-of-thought
    "role",              # expert CFA persona
    "self_consistency",  # SC_SAMPLES majority vote
    "meta",              # explicit elimination strategy
]

SC_SAMPLES      = 3     # self-consistency samples per question
BATCH_SIZE_API  = 5     # questions sent per API call (concurrent)
MODEL_ID        = "qwen/qwen3-32b"
MAX_TOKENS_OUT  = 25    # 25 tokens: enough for non-English letter + surrounding text
MAX_TOKENS_COT  = 512   # CoT / meta strategies need more

_TEMPERATURE_GREEDY = 0.0   # deterministic for most strategies
_TEMPERATURE_SC     = 0.6   # diversity for self-consistency voting
_SC_STRATEGY        = "self_consistency"

print(f"RANDOM_SEED      : {RANDOM_SEED}")
print(f"TOTAL_ROWS_TARGET: {TOTAL_ROWS_TARGET}")
print(f"STRATEGIES       : {STRATEGIES}")
print(f"SC_SAMPLES       : {SC_SAMPLES}")
print(f"BATCH_SIZE_API   : {BATCH_SIZE_API}")
print(f"MODEL            : {MODEL_ID}")
print(f"MAX_TOKENS_OUT   : {MAX_TOKENS_OUT}")
print(f"_TEMPERATURE_GREEDY: {_TEMPERATURE_GREEDY}")
print(f"_TEMPERATURE_SC    : {_TEMPERATURE_SC}")


## Cell 1 — Install Dependencies

In [ ]:
!pip install groq requests nest_asyncio

## Cell 2 — Groq Key Pool

In [ ]:
import asyncio
import time
from collections import deque

# Load keys from GROQ_API.py (Kaggle: upload as dataset) or use inline fallback
try:
    import importlib.util, os
    _spec = importlib.util.spec_from_file_location("GROQ_API", "/kaggle/working/GROQ_API.py")
    _mod  = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_mod)
    GROQ_API = _mod.GROQ_API
    print(f"Loaded {len(GROQ_API)} keys from GROQ_API.py")
except Exception:
    GROQ_API = [
        "YOUR_GROQ_API_KEY_1",
        "YOUR_GROQ_API_KEY_2",
        "YOUR_GROQ_API_KEY_3",
        "YOUR_GROQ_API_KEY_4",
        "YOUR_GROQ_API_KEY_5",
        "YOUR_GROQ_API_KEY_6",
        "YOUR_GROQ_API_KEY_7",
        "YOUR_GROQ_API_KEY_8",
        "YOUR_GROQ_API_KEY_9",
    ]
    print(f"Loaded {len(GROQ_API)} keys (inline fallback)")


class KeyPool:
    """
    Async-safe pool of Groq API keys with per-key sliding-window tracking for:
      RPM  : 60 requests per 60-second window
      TPM  : 6000 tokens  per 60-second window
      RPD  : 1000 requests per calendar day (UTC)
    """

    RPM_LIMIT = 60
    TPM_LIMIT = 6000
    RPD_LIMIT = 1000
    WINDOW    = 60.0

    def __init__(self, keys: list):
        self._keys   = list(keys)
        self._lock   = asyncio.Lock()
        self._rpm_ts = {k: deque() for k in self._keys}
        self._tpm_ts = {k: deque() for k in self._keys}
        self._rpd    = {k: ("", 0) for k in self._keys}

    def _trim_windows(self, key: str, now: float):
        cutoff = now - self.WINDOW
        while self._rpm_ts[key] and self._rpm_ts[key][0] < cutoff:
            self._rpm_ts[key].popleft()
        while self._tpm_ts[key] and self._tpm_ts[key][0][0] < cutoff:
            self._tpm_ts[key].popleft()

    def _is_available(self, key: str, now: float, est_tokens: int) -> bool:
        self._trim_windows(key, now)
        if len(self._rpm_ts[key]) >= self.RPM_LIMIT:
            return False
        used_tpm = sum(t for _, t in self._tpm_ts[key])
        if used_tpm + est_tokens > self.TPM_LIMIT:
            return False
        today = time.strftime("%Y-%m-%d", time.gmtime(now))
        date_str, count = self._rpd[key]
        if date_str == today and count >= self.RPD_LIMIT:
            return False
        return True

    async def get_available_key(self, est_tokens: int = 200) -> str:
        while True:
            async with self._lock:
                now = time.time()
                for key in self._keys:
                    if self._is_available(key, now, est_tokens):
                        return key
            await asyncio.sleep(1)

    def record_use(self, key: str, tokens_used: int = 0):
        now = time.time()
        self._trim_windows(key, now)
        self._rpm_ts[key].append(now)
        if tokens_used > 0:
            self._tpm_ts[key].append((now, tokens_used))
        today = time.strftime("%Y-%m-%d", time.gmtime(now))
        date_str, count = self._rpd[key]
        if date_str == today:
            self._rpd[key] = (today, count + 1)
        else:
            self._rpd[key] = (today, 1)

    def status(self) -> None:
        """Print current rolling quota usage for every key."""
        now   = time.time()
        today = time.strftime("%Y-%m-%d", time.gmtime(now))
        print(f"  {'Key':>12}  {'RPM':>9}  {'TPM':>12}  {'RPD':>13}")
        print("  " + "-" * 52)
        for key in self._keys:
            self._trim_windows(key, now)
            rpm_used = len(self._rpm_ts[key])
            tpm_used = sum(t for _, t in self._tpm_ts[key])
            date_str, rpd_count = self._rpd[key]
            rpd_used = rpd_count if date_str == today else 0
            print(
                f"  ...{key[-8:]:>9}  "
                f"{rpm_used:>3}/{self.RPM_LIMIT:<3}  "
                f"{tpm_used:>5}/{self.TPM_LIMIT:<5}  "
                f"{rpd_used:>5}/{self.RPD_LIMIT}"
            )


KEY_POOL = KeyPool(GROQ_API)
print(f"KeyPool ready — {len(GROQ_API)} keys")
print(f"  Limits per key: RPM={KeyPool.RPM_LIMIT}, "
      f"TPM={KeyPool.TPM_LIMIT}, RPD={KeyPool.RPD_LIMIT}")

## Cell 3 — Load Datasets from GitHub

After loading all datasets, **hindi_finance is capped** so that the total across all
datasets equals exactly `TOTAL_ROWS_TARGET = 2500`. All other datasets are untouched.

In [ ]:
import requests
import json
import base64
import os

_GITHUB_REPO   = "hassan09070/clef_task"
_GITHUB_FOLDER = "task1_data"
_GITHUB_TOKEN  = "YOUR_GITHUB_TOKEN_HERE"


def _get_github_token() -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("github_token")
        if t:
            return t
    except Exception:
        pass
    return os.environ.get("GITHUB_TOKEN") or _GITHUB_TOKEN


def _github_fetch(filename: str) -> list:
    token   = _get_github_token()
    headers = {
        "Accept":        "application/vnd.github+json",
        "Authorization": f"token {token}",
    }
    api_url = (
        f"https://api.github.com/repos/{_GITHUB_REPO}/contents/"
        f"{_GITHUB_FOLDER}/{filename}"
    )
    meta = requests.get(api_url, headers=headers, timeout=60)
    meta.raise_for_status()
    data = meta.json()

    content_b64 = data.get("content", "").replace("\n", "")
    if content_b64:
        return json.loads(base64.b64decode(content_b64).decode("utf-8"))

    download_url = data.get("download_url")
    if not download_url:
        raise ValueError(f"No content or download_url for {filename}")
    print(f"    [large file] streaming {filename} ...")
    chunks, total = [], 0
    raw = requests.get(
        download_url,
        headers={"Authorization": f"token {token}"},
        timeout=300, stream=True,
    )
    raw.raise_for_status()
    for chunk in raw.iter_content(chunk_size=65536):
        chunks.append(chunk)
        total += len(chunk)
    print(f"    [large file] {total // 1024} KB downloaded")
    return json.loads(b"".join(chunks).decode("utf-8"))


_DATASET_FILES = {
    "cfa_cpa":           "task1_Tomas08119993_finmmeval-cfa-cpa.json",
    "es_multifin":       "task1_TheFinAI_flare-es-multifin.json",
    "plutus":            "task1_TheFinAI_plutus-multifin.json",
    "arabic_accounting": "task1_SahmBenchmark_arabic-accounting-mcq.json",
    "arabic_business":   "task1_SahmBenchmark_arabic-business-mcq.json",
    "hindi_finance":     "task1_bharatgenai_BhashaBench-Finance-Hindi.json",
}

_SOURCE_LANG = {
    "cfa_cpa":           "English",
    "es_multifin":       "Spanish",
    "plutus":            "Multilingual",
    "arabic_accounting": "Arabic",
    "arabic_business":   "Arabic",
    "hindi_finance":     "Hindi",
}


def _parse_records(records: list, source_name: str) -> list:
    rows = []
    for rec in records:
        options = rec.get("options") or {}
        if not options:
            continue
        sorted_keys = sorted(options.keys())
        choices     = [options[k] for k in sorted_keys]
        gold_raw    = rec.get("gold") or []
        valid_gold  = [g.lower() for g in gold_raw if g.lower() in sorted_keys]
        if not valid_gold:
            continue
        rows.append({
            "question": str(rec.get("question") or ""),
            "choices":  choices,
            "keys":     sorted_keys,
            "gold":     valid_gold,
            "source":   source_name,
        })
    return rows


# ── Load all datasets ──────────────────────────────────────────────────────
print(f"Loading datasets from github.com/{_GITHUB_REPO} ...")
print()

datasets_raw = {}   # {source_name: full_list_of_rows} — used for few-shot pool

NON_HINDI = [src for src in _DATASET_FILES if src != "hindi_finance"]

for src_name in NON_HINDI:
    fname = _DATASET_FILES[src_name]
    print(f"  Loading {src_name} ...", flush=True)
    rows = _parse_records(_github_fetch(fname), src_name)
    datasets_raw[src_name] = rows
    print(f"  {src_name:<33} {len(rows):>5} rows  OK")

# Load hindi_finance last so we can compute the cap
other_total   = sum(len(v) for v in datasets_raw.values())
hindi_cap     = max(0, TOTAL_ROWS_TARGET - other_total)
print()
print(f"  Other datasets total : {other_total} rows")
print(f"  hindi_finance cap    : {hindi_cap} rows  (to reach {TOTAL_ROWS_TARGET} total)")
print()

print(f"  Loading hindi_finance ...", flush=True)
hindi_rows = _parse_records(_github_fetch(_DATASET_FILES["hindi_finance"]), "hindi_finance")
hindi_rows = hindi_rows[:hindi_cap]
datasets_raw["hindi_finance"] = hindi_rows
print(f"  {'hindi_finance':<33} {len(hindi_rows):>5} rows  OK (capped)")

# Flatten
all_data = []
for src in _DATASET_FILES:          # preserve insertion order
    all_data.extend(datasets_raw[src])

print()
print(f"Total loaded: {len(all_data)} rows across {len(datasets_raw)} datasets")
assert len(all_data) <= TOTAL_ROWS_TARGET, f"Expected <= {TOTAL_ROWS_TARGET}, got {len(all_data)}"
print(f"Row count assertion passed: {len(all_data)} <= {TOTAL_ROWS_TARGET}")

## Cell 4 — Dataset Stats

In [ ]:
from collections import Counter

print(f"Total examples : {len(all_data)}")
print(f"Target         : {TOTAL_ROWS_TARGET}")
print()
print(f"{'Source':<33}  {'Lang':<13}  {'Rows':>5}  Choice distribution")
print("-" * 78)
for src in _DATASET_FILES:
    rows      = datasets_raw[src]
    n_choices = Counter(len(r["choices"]) for r in rows)
    choices_s = "  ".join(f"{k}opt:{v}" for k, v in sorted(n_choices.items()))
    lang      = _SOURCE_LANG.get(src, "?")
    capped    = "  ← capped" if src == "hindi_finance" else ""
    print(f"  {src:<31}  {lang:<13}  {len(rows):>5}  ({choices_s}){capped}")

multi_gold = sum(1 for r in all_data if len(r["gold"]) > 1)
print(f"\nMulti-gold rows: {multi_gold}")

## Cell 5 — Prompt Builders

In [ ]:
import re

# Seed hardcoded inline — safe to run this cell independently of Cell 0
_FEWSHOT_RNG = random.Random(42)


def _labels_and_options(choices: list) -> tuple:
    n      = len(choices)
    labels = [chr(ord("A") + i) for i in range(n)]
    opts   = "\n".join(f"{labels[i]}. {choices[i]}" for i in range(n))
    valid  = "/".join(labels)
    return labels, opts, valid


def _extract_letter(text: str, valid_labels: list) -> str | None:
    text_up = text.upper()
    for pat in [r"ANSWER\s*:\s*([A-Z])", r"THE ANSWER IS\s*([A-Z])", r"FINAL ANSWER\s*:\s*([A-Z])"]:
        m = re.search(pat, text_up)
        if m and m.group(1) in valid_labels:
            return m.group(1).lower()
    for pat in [r"[.\n]\s*([A-Z])\s*$", r"\*\*([A-Z])\*\*"]:
        m = re.search(pat, text_up)
        if m and m.group(1) in valid_labels:
            return m.group(1).lower()
    for ch in text_up:
        if ch in valid_labels:
            return ch.lower()
    return None


def _build_few_shot_examples(source: str, exclude_idx: int, n: int = 3) -> str:
    pool = [
        r for i, r in enumerate(all_data)
        if r["source"] == source and i != exclude_idx
    ]
    if len(pool) < n:
        pool = [r for i, r in enumerate(all_data) if i != exclude_idx]
    sampled = _FEWSHOT_RNG.sample(pool, min(n, len(pool)))
    parts   = []
    for ex in sampled:
        labels, opts, _ = _labels_and_options(ex["choices"])
        answer_letter   = ex["gold"][0].upper()
        parts.append(f"Question: {ex['question']}\n{opts}\nAnswer: {answer_letter}")
    return "\n\n".join(parts)


_ROLE_SYSTEM = (
    "You are a senior Chartered Financial Analyst (CFA) with over 20 years of "
    "experience in financial markets, accounting standards, and investment analysis. "
    "You have deep expertise in international finance, business law, and economics. "
    "When answering questions, draw on your professional expertise and provide the "
    "most accurate financial judgement."
)


def _make_messages(system: str, user: str) -> list:
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": user})
    return msgs


# ── Prompt builders ────────────────────────────────────────────────────────

def build_baseline_prompt(question, choices, **_):
    labels, opts, valid = _labels_and_options(choices)
    user = (
        "Answer the following multiple-choice question. "
        f"Respond with ONLY the single letter of the correct answer ({valid}).\n\n"
        f"Question: {question}\n\n{opts}"
    )
    return _make_messages("", user), MAX_TOKENS_OUT


def build_few_shot_prompt(question, choices, source, row_idx, **_):
    labels, opts, valid = _labels_and_options(choices)
    examples = _build_few_shot_examples(source, row_idx, n=3)
    user = (
        "Answer financial multiple-choice questions. "
        f"Respond with ONLY the single letter ({valid}).\n\n"
        "Here are three solved examples:\n\n"
        f"{examples}\n\n"
        "Now answer this question:\n\n"
        f"Question: {question}\n\n{opts}\nAnswer:"
    )
    return _make_messages("", user), MAX_TOKENS_OUT


def build_cot_prompt(question, choices, **_):
    labels, opts, valid = _labels_and_options(choices)
    user = (
        "Answer the following financial multiple-choice question. "
        "Think through the problem step by step, then state your final answer "
        f"on a new line as: Answer: <letter>  (where letter is one of {valid}).\n\n"
        f"Question: {question}\n\n{opts}"
    )
    return _make_messages("", user), MAX_TOKENS_COT


def build_role_prompt(question, choices, **_):
    labels, opts, valid = _labels_and_options(choices)
    user = (
        f"Answer this multiple-choice question. Respond with ONLY the letter ({valid}).\n\n"
        f"Question: {question}\n\n{opts}"
    )
    return _make_messages(_ROLE_SYSTEM, user), MAX_TOKENS_OUT


def build_meta_prompt(question, choices, **_):
    labels, opts, valid = _labels_and_options(choices)
    user = (
        "You will answer a financial multiple-choice question. "
        "Use the following reasoning strategy:\n"
        "1. Read all options carefully.\n"
        "2. Eliminate options that are clearly incorrect.\n"
        "3. For the remaining options, identify which one is most precisely correct "
        "according to standard financial principles.\n"
        "4. State your final answer as: Answer: <letter>\n\n"
        f"Valid answer letters: {valid}\n\n"
        f"Question: {question}\n\n{opts}"
    )
    return _make_messages("", user), MAX_TOKENS_COT


def build_self_consistency_prompt(question, choices, **_):
    return build_baseline_prompt(question, choices)


PROMPT_BUILDERS = {
    "baseline":         build_baseline_prompt,
    "few_shot":         build_few_shot_prompt,
    "cot":              build_cot_prompt,
    "role":             build_role_prompt,
    "self_consistency": build_self_consistency_prompt,
    "meta":             build_meta_prompt,
}

print("Prompt builders registered:")
for name in PROMPT_BUILDERS:
    print(f"  {name}")

## Cell 6 — Async Groq Inference Engine

- Each API call checks out a key via `KeyPool.get_available_key()` and records the
  actual tokens used via `KeyPool.record_use()` after the response.
- `BATCH_SIZE_API` questions are dispatched concurrently with `asyncio.gather`.
- Self-consistency fires `SC_SAMPLES` concurrent calls per question and majority-votes.

In [ ]:
from groq import AsyncGroq

async def _call_api(
    messages: list,
    max_tokens: int,
    temperature: float = _TEMPERATURE_GREEDY,
    retries: int = 5,
) -> str:
    """Single Groq API call with exponential-backoff retry."""
    est_tokens = sum(len(m["content"]) // 4 for m in messages) + max_tokens
    for attempt in range(retries):
        try:
            key    = await KEY_POOL.get_available_key(est_tokens=est_tokens)
            client = AsyncGroq(api_key=key)
            resp   = await client.chat.completions.create(
                model=MODEL_ID,
                messages=messages,
                max_completion_tokens=max_tokens,
                temperature=temperature,
                top_p=1,
                stream=False,
                stop=["\n", "Explanation"] if max_tokens <= MAX_TOKENS_OUT else None,
            )
            actual_tokens = getattr(resp.usage, "total_tokens", est_tokens)
            KEY_POOL.record_use(key, tokens_used=actual_tokens)
            return resp.choices[0].message.content or ""
        except Exception as e:
            wait = 2 ** attempt
            print(f"  [retry {attempt+1}/{retries}] {type(e).__name__}: {e} — wait {wait}s", flush=True)
            await asyncio.sleep(wait)
    return ""  # all retries exhausted — caller will fall back to keys[0]


async def _predict_one(row: dict, row_idx: int, strategy: str) -> str:
    """Predict a single question with the given strategy."""
    builder = PROMPT_BUILDERS[strategy]
    messages, max_tokens = builder(
        question=row["question"],
        choices=row["choices"],
        source=row["source"],
        row_idx=row_idx,
    )
    labels = [chr(ord("A") + i) for i in range(len(row["choices"]))]

    text     = await _call_api(messages, max_tokens)
    pred_key = _extract_letter(text, labels)
    if pred_key is None:
        pred_key = row["keys"][0]   # fallback: first option
    return pred_key


async def _predict_sc(row: dict, row_idx: int) -> str:
    """Self-consistency: fire SC_SAMPLES calls concurrently and majority-vote."""
    builder = PROMPT_BUILDERS["self_consistency"]
    messages, max_tokens = builder(
        question=row["question"],
        choices=row["choices"],
        source=row["source"],
        row_idx=row_idx,
    )
    labels = [chr(ord("A") + i) for i in range(len(row["choices"]))]

    texts = await asyncio.gather(*[
        _call_api(messages, max_tokens, temperature=_TEMPERATURE_SC)
        for _ in range(SC_SAMPLES)
    ])

    from collections import Counter
    votes = [_extract_letter(t, labels) for t in texts]
    votes = [v for v in votes if v is not None]
    if votes:
        return Counter(votes).most_common(1)[0][0]
    return row["keys"][0]   # fallback


async def run_strategy_async(data: list, strategy: str) -> tuple:
    """
    Run strategy over all data with BATCH_SIZE_API concurrent requests.
    Returns (predictions, errors).
    """
    n     = len(data)
    preds = [None] * n
    print(f"  [{strategy}] starting — {n} questions  (batch={BATCH_SIZE_API} concurrent)", flush=True)

    for batch_start in range(0, n, BATCH_SIZE_API):
        batch = data[batch_start: batch_start + BATCH_SIZE_API]

        if strategy == _SC_STRATEGY:
            tasks = [
                _predict_sc(row, batch_start + i)
                for i, row in enumerate(batch)
            ]
        else:
            tasks = [
                _predict_one(row, batch_start + i, strategy)
                for i, row in enumerate(batch)
            ]

        results = await asyncio.gather(*tasks)
        for k, p in enumerate(results):
            preds[batch_start + k] = p

        done    = batch_start + len(batch)
        correct = sum(1 for j in range(done) if preds[j] in data[j]["gold"])
        print(f"  [{strategy}] [{done:>5}/{n}]  running acc: {correct/done*100:.1f}%", flush=True)

    correct = sum(1 for i, p in enumerate(preds) if p in data[i]["gold"])
    acc     = correct / n * 100
    print(f"  [{strategy}] FINAL — acc: {acc:.2f}%  ({correct}/{n})", flush=True)

    errors = [
        {
            "idx":       i,
            "source":    data[i]["source"],
            "gold":      data[i]["gold"],
            "predicted": preds[i],
            "strategy":  strategy,
        }
        for i, p in enumerate(preds)
        if p not in data[i]["gold"]
    ]
    return preds, errors


print("Async inference engine ready.")
print(f"  BATCH_SIZE_API : {BATCH_SIZE_API} concurrent requests")
print(f"  SC_SAMPLES     : {SC_SAMPLES} samples per SC question")


## Cell 7 — Run All Strategies

In [ ]:
import time
import json as _json
import os

_CHECKPOINT_FILE = "checkpoint_groq_qwen3.json"

all_results = {}
all_errors  = {}

# Load existing checkpoint so a crash-restart resumes where it left off
if os.path.exists(_CHECKPOINT_FILE):
    print(f"Found checkpoint: {_CHECKPOINT_FILE}")
    with open(_CHECKPOINT_FILE) as _f:
        _ckpt = _json.load(_f)
    for _strat, _data in _ckpt.items():
        all_results[_strat] = {
            "predictions": _data["predictions"],
            "accuracy":    _data["accuracy"],
        }
        all_errors[_strat] = _data.get("errors", [])
        print(f"  Restored [{_strat}]  acc={_data['accuracy']:.2f}%")
    print()
else:
    print("No checkpoint found - starting fresh.")
print()


def _save_checkpoint():
    """Persist all completed strategies to disk immediately."""
    ckpt = {}
    for strat, res in all_results.items():
        ckpt[strat] = {
            "predictions": res["predictions"],
            "accuracy":    res["accuracy"],
            "errors":      all_errors.get(strat, []),
        }
    with open(_CHECKPOINT_FILE, "w") as _cf:
        _json.dump(ckpt, _cf)
    n = len(ckpt)
    label = "strategy" if n == 1 else "strategies"
    print(f"  [checkpoint] saved {n} {label} to {_CHECKPOINT_FILE}", flush=True)


async def _run_all():
    for strategy in STRATEGIES:
        if strategy in all_results:
            print(f"  [SKIP] {strategy} - already in checkpoint", flush=True)
            continue

        print("=" * 65)
        print(f"STRATEGY: {strategy}")
        print(f"  questions : {len(all_data)}")
        print("=" * 65)

        t0            = time.time()
        preds, errors = await run_strategy_async(all_data, strategy)
        elapsed       = time.time() - t0

        correct  = sum(1 for i, p in enumerate(preds) if p in all_data[i]["gold"])
        accuracy = correct / len(all_data) * 100

        all_results[strategy] = {"predictions": preds, "accuracy": accuracy}
        all_errors[strategy]  = errors

        print(f"  Elapsed : {elapsed:.0f}s")
        print(f"  Overall accuracy [{strategy}]: {accuracy:.2f}%  ({correct}/{len(all_data)})")
        print(f"  Wrong predictions stored: {len(errors)}")

        # Checkpoint immediately - safe even if the next strategy crashes
        _save_checkpoint()

        # Key pool quota snapshot
        print(f"\n  Key pool status after [{strategy}]:")
        KEY_POOL.status()
        print()


# Apply nest_asyncio so this works in Jupyter/Kaggle and plain terminal
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

asyncio.get_event_loop().run_until_complete(_run_all())

print("=" * 65)
print("ALL STRATEGIES COMPLETE")
print("=" * 65)
for strat, res in all_results.items():
    n_err = len(all_errors.get(strat, []))
    print(f"  {strat:<22}  {res['accuracy']:.2f}%   ({n_err} errors logged)")

## Cell 8 — Results: Cartesian Comparison (Strategy × Dataset)

In [ ]:
import pandas as pd

sources     = list(_DATASET_FILES.keys())
src_indices = {
    src: [i for i, r in enumerate(all_data) if r["source"] == src]
    for src in sources
}

table_rows = []
for strategy, res in all_results.items():
    preds    = res["predictions"]
    row_dict = {"strategy": strategy}
    for src in sources:
        idxs          = src_indices[src]
        c             = sum(1 for i in idxs if preds[i] in all_data[i]["gold"])
        row_dict[src] = round(c / len(idxs) * 100, 1) if idxs else 0.0
    c_all               = sum(1 for i, p in enumerate(preds) if p in all_data[i]["gold"])
    row_dict["OVERALL"] = round(c_all / len(all_data) * 100, 1)
    table_rows.append(row_dict)

df_cart = pd.DataFrame(table_rows).set_index("strategy")

print("=" * 80)
print("CARTESIAN RESULTS: Accuracy (%)  —  Strategy × Dataset")
print("=" * 80)
print(df_cart.to_string())
print()

print("-" * 80)
print("Strategy ranking by OVERALL accuracy:")
ranked = df_cart["OVERALL"].sort_values(ascending=False)
for rank, (strat, acc) in enumerate(ranked.items(), 1):
    marker = "  ◄ best" if rank == 1 else ""
    print(f"  {rank}. {strat:<22}  {acc:.1f}%{marker}")


print()
print("-" * 80)
print("Best strategy per dataset:")
for src in sources:
    if src not in df_cart.columns:
        continue
    best_strat = df_cart[src].idxmax()
    best_acc   = df_cart[src].max()
    print(f"  {src:<31}  {best_strat:<22}  {best_acc:.1f}%")
if "OVERALL" in df_cart.columns:
    best_overall_strat = df_cart["OVERALL"].idxmax()
    best_overall_acc   = df_cart["OVERALL"].max()
    print(f"  {'OVERALL':<31}  {best_overall_strat:<22}  {best_overall_acc:.1f}%")

try:
    import seaborn as sns
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(max(8, len(sources) * 1.4), max(4, len(df_cart) * 0.7)))
    sns.heatmap(
        df_cart, annot=True, fmt=".1f", cmap="RdYlGn",
        linewidths=0.5, linecolor="white",
        vmin=max(0, df_cart.values.min() - 5),
        vmax=min(100, df_cart.values.max() + 5),
        ax=ax,
    )
    ax.set_title("Accuracy (%) — Strategy × Dataset  [Groq qwen3-32b]", fontsize=13, pad=12)
    ax.set_xlabel("Dataset / OVERALL", fontsize=10)
    ax.set_ylabel("Prompt Strategy", fontsize=10)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig("groq_qwen3_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Heatmap saved: groq_qwen3_heatmap.png")
except ImportError:
    print("[heatmap] pip install seaborn matplotlib")

## Cell 9 — Save Results

In [ ]:
import json

output = {}
for strategy, res in all_results.items():
    preds   = res["predictions"]
    correct = sum(1 for i, p in enumerate(preds) if p in all_data[i]["gold"])
    src_accs = {}
    for src in sources:
        idxs = src_indices[src]
        c    = sum(1 for i in idxs if preds[i] in all_data[i]["gold"])
        src_accs[src] = round(c / len(idxs) * 100, 2) if idxs else 0.0
    output[strategy] = {
        "accuracy":    res["accuracy"],
        "correct":     int(correct),
        "total":       len(all_data),
        "per_source":  src_accs,
        "predictions": preds,
    }

fname_json = "groq_qwen3_prompt_engineering_results.json"
with open(fname_json, "w") as f:
    json.dump(output, f, indent=2)
print(f"Saved: {fname_json}")

fname_csv = "groq_qwen3_prompt_engineering_cartesian.csv"
df_cart.reset_index().to_csv(fname_csv, index=False)
print(f"Saved: {fname_csv}")

error_rows = []
for strategy, errs in all_errors.items():
    for e in errs:
        row_data = all_data[e["idx"]]
        error_rows.append({
            "strategy":  strategy,
            "idx":       e["idx"],
            "source":    e["source"],
            "language":  _SOURCE_LANG.get(e["source"], "?"),
            "gold":      "|".join(e["gold"]),
            "predicted": e["predicted"],
            "question":  row_data["question"][:200],
        })

fname_errors = "groq_qwen3_prompt_engineering_errors.csv"
pd.DataFrame(error_rows).to_csv(fname_errors, index=False)
print(f"Saved: {fname_errors}  ({len(error_rows)} error rows)")
print()
print("Files written:")
print(f"  {fname_json}")
print(f"  {fname_csv}")
print(f"  {fname_errors}")